<a href="https://colab.research.google.com/github/pazolka/ai-climate-workshop-dhaka/blob/main/5.%20Water%20quality%20group%20work/Salinity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pazolka/ai-climate-workshop-dhaka.git

# Group work - river and soil salinity data

**Due to the time constraints, these will be optional exercises to be completed in your own time. Please send an email to Anna anna.pazola.20@ucl.ac.uk if you have any questions / results that you want to discuss :)**

This session relies on the Soil Resource Development Institute (SRDI) river and soil salinity data that have been preprocessed to alter the raw original values. This doesn't affect the analysis. _The original data can be requested directly from SRDI_. We also use ERA5 reanalysis data (precipitation, evapotranspiration, temperature, soil moisture) that was obtained from Copernicus Climate Data Store and sampled at the salinity observation points. TerraClimate precipitation was downloaded from Google Earth Engine.

Working in small groups, choose one of the exercises to complete.

1. *Data imputation* - similar the the GWL examples from our first session, can we fill the gaps in the salinity time series? You have an option to use station data only, or other covariates.

2. *Rainfall mission* - create a predictive model and investigate which rainfall dataset explains the salinity time series better.

3. *Evapotranspiration and soil salinity mission* - create a predictive model and investigate whether other predictors beyond rainfall add predictive skill.

Produce one graph and one conclusion basd on your analysis.

In [ ]:
# common libraries to import

import warnings
warnings.filterwarnings('ignore', message='.*Google will stop supporting.*')
warnings.filterwarnings('ignore', category=UserWarning)

import sys
sys.path.append('/content/ai-climate-workshop-dhaka/5. Water quality group work')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

import helpers as hlp

# shared train/test split date used by the rainfall and evaporation exercises
TRAIN_END = pd.Period('2016-12', 'M')


In [ ]:
# read the data
SAL_PATH = 'ai-climate-workshop-dhaka/data/point_observations/SRDI_water_soil_salinity_preprocessed.csv'
STATION_PATH = 'ai-climate-workshop-dhaka/data/point_observations/SRDI_salinity_station_info.csv'
DRIVERS_PATH = 'ai-climate-workshop-dhaka/data/point_observations/ERA5_Land_salinity_station_monthly_2004_2022.csv'
ERA5_PATH = 'ai-climate-workshop-dhaka/data/gridded/ERA5_precip_M_acc_1972_2022_BGD_025.nc'
TC_PATH = 'ai-climate-workshop-dhaka/data/gridded/TerraClimate_precip_aet_M_1972_2022_BGD_005.nc'

stations = pd.read_csv(STATION_PATH)

salinity_ts = pd.read_csv(SAL_PATH)
salinity_ts.index = pd.to_datetime({'year': salinity_ts['Year'], 'month': salinity_ts['Month'], 'day': 1})
salinity_ts.index = salinity_ts.index.to_period('M')
salinity_ts = salinity_ts[stations.Station]

drivers = pd.read_csv(DRIVERS_PATH, parse_dates=['date'])
drivers['date'] = drivers['date'].dt.to_period('M')

print('salinity:', salinity_ts.shape, salinity_ts.index.min(), '..', salinity_ts.index.max())
print('drivers :', drivers.shape)
drivers.head()

### 1. Data imputation with Random Forest

The salinity time series has many missing months. As we have seen in one of the previous sessions, Random Forest - based imputation techniques are applicable even to very small datasets. In this exercise, apply the same technique (`IterativeImputer` + `RandomForestRegressor`) to the salinity time series.

**Your choice:** your input data are either

- **stations only** — the 24 salinity series; or
- **stations + rainfall / temperature / soil moisture / evapotranspiration** — the other series plus that station's ERA5 monthly covariates.

**Evaluation:** Choose at least two input settings. For comparison, hide a random fraction of the observed values, fill the gaps, and score the hidden values (RMSE and MAE). Lower is better.

In [ ]:
# rainfall at each station, aligned to the salinity table
# (swap 'precipitation' for 'temperature', 'soil_moisture' or 'evaporation')
rain_wide = (drivers.pivot(index='date', columns='Station', values='precipitation')
             .reindex(index=salinity_ts.index, columns=salinity_ts.columns)
             .add_suffix('_rain'))

# hide 10 % of the observed salinity values at random
rng = np.random.default_rng(0)
observed = salinity_ts.notna().values.ravel()
# select 10% of the observed values to hide at random
hide = rng.choice(np.flatnonzero(observed), size=int(0.1 * observed.sum()), replace=False)
# create a boolean mask of the same shape as salinity_ts, with True for the hidden values
mask_values = np.zeros(salinity_ts.shape, dtype=bool)
mask_values.ravel()[hide] = True
mask = pd.DataFrame(mask_values, index=salinity_ts.index, columns=salinity_ts.columns)

# options:
# Option 1) stations only: X = salinity_ts.copy()
# Option 2) stations + rainfall : X = pd.concat([salinity_ts, rain_wide], axis=1)
# For other covariates, build the same wide table with values='temperature',
# 'soil_moisture' or 'evaporation'.

# Option 1) BASELINE
X = [YOUR OPTION HERE]

# set the hidden values to NaN in X, so that the imputer will try to fill them
X[mask] = np.nan

imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=100, max_features='sqrt',
                                    min_samples_leaf=5, n_jobs=-1, random_state=42),
    max_iter=10, random_state=42)

filled = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
truth, pred = salinity_ts.values[mask.values], filled.values[mask.values]

print(f'Option 1 | rmse: {np.sqrt(np.mean((truth - pred) ** 2))}')
print(f'Option 1 | mae: {np.mean(np.abs(truth - pred))}')

# Option 2) WITH COVARIATES 
X = [YOUR OPTION HERE]
# set the hidden values to NaN in X, so that the imputer will try to fill them
X[mask.reindex(columns=X.columns, fill_value=False)] = np.nan

filled = [YOUR CODE HERE]
truth, pred = salinity_ts.values[mask.values], filled[salinity_ts.columns].values[mask.values]

print(f'Option 2 | rmse: {np.sqrt(np.mean((truth - pred) ** 2))}')
print(f'Option 2 | mae: {np.mean(np.abs(truth - pred))}')


### 2. Rainfall mission: TerraClimate vs ERA5

Does TerraClimate explain salinity better than ERA5?

Absolute salinity levels differ strongly between stations, so each station's salinity is converted to an anomaly (subtract its mean, similar to GWSA modelling session); this lets the pooled model learn the temporal rainfall response rather than station identity, and the mean is added back only when computing RMSE.

**Your choices:**

- station subset: `all`, `river`, or `soil`;
- lag: 0, 1, 2 or 3 months (rainfall earlier by that many months);
- model: Random Forest, Gaussian Process, or XGBoost.

**Evaluation** Each model is trained on 2004-2016 and tested on 2017-2022, and is fitted on all station-months in the subset, using rainfall at every month from the current month (lag 0) through the chosen lag (e.g. `LAG=2` uses current, 1-month and 2-month lag). RMSE is reported in µS/cm.

In [ ]:
# feel free to choose a different subset of stations, or a different lag, to see how it affects the results
SUBSET = 'all'      # 'all', 'river' or 'soil'
LAG = 5             # 0, 1, 2, 3 ...

# sample both rainfall products at the stations and build one long panel
panel = hlp.build_panel(salinity_ts, stations, ERA5_PATH, TC_PATH)

# keep only the station type chosen above
if SUBSET != 'all':
    panel = panel[panel.type == SUBSET].copy()

# add every rainfall lag from the current month up to LAG
panel, era5_features, tc_features = hlp.add_lags(panel, LAG)

# compute the salinity anomaly for each station - this is your target variable!
mean_by_station = panel[panel.date <= TRAIN_END].groupby('Station')['salinity'].mean()
panel['sal_anom'] = panel.salinity - panel.Station.map(mean_by_station)

cols = ['sal_anom'] + era5_features + tc_features

# divide your data into a training set and a test set, based on the TRAIN_END date defined above
train = panel[panel.date <= TRAIN_END].dropna(subset=cols).copy()
test = panel[panel.date > TRAIN_END].dropna(subset=cols).copy()

# standardise the rainfall features (GP needs this; trees do not, but it does not hurt them)
scaler_era5 = StandardScaler().fit(train[era5_features])
train[era5_features] = scaler_era5.transform(train[era5_features])
test[era5_features] = scaler_era5.transform(test[era5_features])

scaler_tc = StandardScaler().fit(train[tc_features])
train[tc_features] = scaler_tc.transform(train[tc_features])
test[tc_features] = scaler_tc.transform(test[tc_features])

## ::::: Random Forest model
model = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)

## ::::: XGBoost model
# model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, # you can also try different hyperparameters here; refer to the API documentation https://xgboost.readthedocs.io/en/latest/python/python_api.html#module-xgboost.sklearn
#                         colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)

## ::::: Gaussian Process model with a kernel that is the sum of a constant kernel, an RBF kernel and a white noise kernel
# model = GaussianProcessRegressor(
#     kernel=ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(0.1),
#     normalize_y=True, alpha=1e-3, optimizer=None, random_state=42)

# Model 1: ERA5 based
model.fit(train[era5_features], train['sal_anom']) # fit the model to the training data
pred_era5 = model.predict(test[era5_features]) + test.Station.map(mean_by_station) # make predictions on the test data and add back the mean salinity for each station
rmse_era5 = np.sqrt(mean_squared_error(test.salinity, pred_era5)) # compute the root mean squared error of the predictions

# Model 2: TerraClimate based
model.fit(train[tc_features], train['sal_anom'])
pred_tc = model.predict(test[tc_features]) + test.Station.map(mean_by_station)
rmse_tc = np.sqrt(mean_squared_error(test.salinity, pred_tc))

# a small summary table
pd.DataFrame({'product': ['ERA5', 'TerraClimate'],
              'n_train': [len(train), len(train)],
              'n_test': [len(test), len(test)],
              'rmse': [rmse_era5, rmse_tc]}).round(1)

In [ ]:
# Example plotting the observed and predicted salinity for a specific station
test['salinity_pred'] = pred_tc
station = test[test.Station == 'Morichap'].set_index('date')

station['salinity'].plot()
station['salinity_pred'].plot()

### 3. Evaporation/heat mission

Does temperature or soil moisture from ERA5-Land add skill beyond rainfall?

**Your choices:**

- station subset: `all`, `river`, or `soil`;
- model: Random Forest, Gaussian Process, or XGBoost;
- extra variables: rain + temperature, rain + soil moisture, rain + evaporation, or rain + all.

Each station's salinity is converted to an anomaly using the training-period mean. The models of your choice are fitted on the pooled station-months. Each model is trained on 2004-2016 and tested on 2017-2022. The table reports RMSE in µS/cm.

In [ ]:
# feel free to choose a different subset, feature set, or model
SUBSET = 'all'                  # 'all', 'river' or 'soil'
ALL_FEATURES = ['precipitation', 'evaporation', 'temperature', 'soil_moisture']

# one long panel: salinity + ERA5-Land drivers
sal_long = salinity_ts.rename_axis(index='date', columns='Station').stack().rename('salinity').reset_index()
panel = sal_long.merge(drivers, on=['Station', 'date'])
panel['type'] = panel.Station.map(stations.set_index('Station')['Type'])
if SUBSET != 'all':
    panel = panel[panel.type == SUBSET].copy()

# target: salinity anomaly, with station means from the training period only
mean_by_station = panel[panel.date <= TRAIN_END].groupby('Station')['salinity'].mean()
panel['sal_anom'] = panel.salinity - panel.Station.map(mean_by_station)

train = panel[panel.date <= TRAIN_END].dropna().copy()
test = panel[panel.date > TRAIN_END].dropna().copy()

# standardise all drivers once (GP needs this; trees do not, but it does not hurt them)
scaler = StandardScaler().fit(train[ALL_FEATURES])
train[ALL_FEATURES] = scaler.transform(train[ALL_FEATURES])
test[ALL_FEATURES] = scaler.transform(test[ALL_FEATURES])

## ::::: Random Forest model
model = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)

## ::::: XGBoost model
# model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8,
#                      colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)

## ::::: Gaussian Process model
# model = GaussianProcessRegressor(
#     kernel=ConstantKernel(1.0) * RBF(length_scale=1.0) + WhiteKernel(0.1),
#     normalize_y=True, alpha=1e-3, optimizer=None, random_state=42)

# Model 1: rainfall only (feel free to try other single variables, e.g. 'temperature', 'soil_moisture', 'evaporation') or a combination of them
FEATURES = ['precipitation']   # choose any subset of the four drivers
model.fit(train[['precipitation']], train['sal_anom']) # choose any subset of the four drivers
pred_rain = model.predict(test[['precipitation']]) + test.Station.map(mean_by_station)
rmse_rain = np.sqrt(mean_squared_error(test.salinity, pred_rain))

# Model 2: rainfall + the chosen extra variables
FEATURES = ['precipitation', 'evaporation', 'temperature']   # choose any subset of the four drivers
model.fit(train[FEATURES], train['sal_anom'])
pred_covs = model.predict(test[FEATURES]) + test.Station.map(mean_by_station)
rmse_features = np.sqrt(mean_squared_error(test.salinity, pred_covs))

# a small summary table
pd.DataFrame({'features': ['rain', 'rain + ' + ' + '.join(FEATURES[1:])],
              'n_train': [len(train), len(train)],
              'n_test': [len(test), len(test)],
              'rmse': [rmse_rain, rmse_features]}).round(1)

In [ ]:
# Example plotting the observed and predicted salinity for a specific station
test['salinity_pred'] = pred_covs
station = test[test.Station == 'Morichap'].set_index('date')

station['salinity'].plot()
station['salinity_pred'].plot()